In [1]:
# ========================
# Week 6 - Day 1: Incident Response
# ========================

In [2]:
import re
from datetime import datetime
from collections import Counter

# Simulate SSH auth log entries (like /var/log/auth.log)
sample_logs = """
Jul 19 10:23:01 server sshd[1234]: Failed password for root from 192.168.1.100 port 22
Jul 19 10:23:02 server sshd[1234]: Failed password for root from 192.168.1.100 port 22
Jul 19 10:23:03 server sshd[1234]: Failed password for root from 192.168.1.100 port 22
Jul 19 10:23:04 server sshd[1234]: Failed password for admin from 192.168.1.100 port 22
Jul 19 10:23:05 server sshd[1234]: Failed password for admin from 192.168.1.100 port 22
Jul 19 10:23:06 server sshd[1234]: Accepted password for sharuk from 192.168.1.200 port 22
Jul 19 10:23:07 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:08 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:09 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:10 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:11 server sshd[1234]: Accepted password for sharuk from 192.168.1.200 port 22
Jul 19 10:23:12 server sshd[1234]: Failed password for guest from 172.16.0.25 port 22
"""

def analyze_logs(logs):
    print(f"\n{'='*55}")
    print(f"🔍 LOG ANALYSIS REPORT")
    print(f"{'='*55}")
    
    lines = logs.strip().split('\n')
    
    failed_attempts = []
    successful_logins = []
    ip_counter = Counter()
    
    for line in lines:
        # Extract failed attempts
        if "Failed password" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                ip = ip_match.group(1)
                user = user_match.group(1)
                failed_attempts.append({"ip": ip, "user": user})
                ip_counter[ip] += 1
        
        # Extract successful logins
        if "Accepted password" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                successful_logins.append({
                    "ip": ip_match.group(1),
                    "user": user_match.group(1)
                })
    
    # Report findings
    print(f"\n📊 SUMMARY")
    print(f"Total log entries:     {len(lines)}")
    print(f"Failed attempts:       {len(failed_attempts)}")
    print(f"Successful logins:     {len(successful_logins)}")
    
    print(f"\n🚨 SUSPICIOUS IPs (3+ failed attempts)")
    for ip, count in ip_counter.most_common():
        if count >= 3:
            print(f"  {ip:20s} → {count} failed attempts ← BRUTE FORCE SUSPECTED")
    
    print(f"\n✅ SUCCESSFUL LOGINS")
    for login in successful_logins:
        print(f"  User: {login['user']:10s} from {login['ip']}")
    
    print(f"\n💡 RECOMMENDATIONS")
    suspicious = [ip for ip, count in ip_counter.items() if count >= 3]
    if suspicious:
        print(f"  Block these IPs immediately:")
        for ip in suspicious:
            print(f"  → sudo ufw deny from {ip}")

analyze_logs(sample_logs)


🔍 LOG ANALYSIS REPORT

📊 SUMMARY
Total log entries:     12
Failed attempts:       10
Successful logins:     2

🚨 SUSPICIOUS IPs (3+ failed attempts)
  192.168.1.100        → 5 failed attempts ← BRUTE FORCE SUSPECTED
  10.0.0.50            → 4 failed attempts ← BRUTE FORCE SUSPECTED

✅ SUCCESSFUL LOGINS
  User: sharuk     from 192.168.1.200
  User: sharuk     from 192.168.1.200

💡 RECOMMENDATIONS
  Block these IPs immediately:
  → sudo ufw deny from 192.168.1.100
  → sudo ufw deny from 10.0.0.50


In [3]:
import subprocess

def analyze_real_logs():
    print(f"\n{'='*55}")
    print(f"🔍 REAL SYSTEM LOG ANALYSIS")
    print(f"{'='*55}")
    
    # Get real SSH logs from your system
    result = subprocess.run(
        ['journalctl', '-u', 'sshd', '--no-pager', '-n', '50'],
        capture_output=True, text=True
    )
    
    logs = result.stdout
    lines = logs.strip().split('\n')
    
    failed = []
    accepted = []
    ip_counter = Counter()
    
    for line in lines:
        if "Failed password" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                ip = ip_match.group(1)
                failed.append({"ip": ip, "user": user_match.group(1)})
                ip_counter[ip] += 1
        
        if "Accepted" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                accepted.append({
                    "ip": ip_match.group(1),
                    "user": user_match.group(1)
                })
    
    print(f"Last 50 SSH log entries analyzed")
    print(f"Failed attempts:   {len(failed)}")
    print(f"Successful logins: {len(accepted)}")
    
    if ip_counter:
        print(f"\n🚨 Failed login IPs:")
        for ip, count in ip_counter.most_common():
            print(f"  {ip} → {count} attempts")
    else:
        print(f"\n✅ No failed login attempts found — system clean!")
    
    if accepted:
        print(f"\n✅ Successful logins:")
        for login in accepted:
            print(f"  {login['user']} from {login['ip']}")

analyze_real_logs()


🔍 REAL SYSTEM LOG ANALYSIS
Last 50 SSH log entries analyzed
Failed attempts:   0
Successful logins: 0

✅ No failed login attempts found — system clean!


In [4]:
import time
from collections import Counter
import re

def real_time_monitor(threshold=3, check_interval=10):
    print(f"🔴 LIVE SSH MONITOR STARTED")
    print(f"Alert threshold: {threshold} failed attempts")
    print(f"Check interval: {check_interval} seconds")
    print(f"Press Ctrl+C to stop\n")
    
    import subprocess
    seen_alerts = set()
    
    try:
        while True:
            # Get recent logs
            result = subprocess.run(
                ['journalctl', '-u', 'sshd', '--no-pager',
                 '--since', '5 minutes ago'],
                capture_output=True, text=True
            )
            
            ip_counter = Counter()
            for line in result.stdout.split('\n'):
                if "Failed password" in line:
                    ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
                    if ip_match:
                        ip_counter[ip_match.group(1)] += 1
            
            # Check for threshold breaches
            for ip, count in ip_counter.items():
                if count >= threshold and ip not in seen_alerts:
                    print(f"🚨 ALERT! {datetime.now().strftime('%H:%M:%S')}")
                    print(f"   Brute force detected from {ip}")
                    print(f"   {count} failed attempts in last 5 minutes")
                    print(f"   Run: sudo ufw deny from {ip}\n")
                    seen_alerts.add(ip)
            
            if not ip_counter:
                print(f"✅ {datetime.now().strftime('%H:%M:%S')} — No threats detected")
            
            time.sleep(check_interval)
            
    except KeyboardInterrupt:
        print(f"\n🔴 Monitor stopped")

# Run for 30 seconds
real_time_monitor(threshold=3, check_interval=10)

🔴 LIVE SSH MONITOR STARTED
Alert threshold: 3 failed attempts
Check interval: 10 seconds
Press Ctrl+C to stop

✅ 19:55:30 — No threats detected
✅ 19:55:40 — No threats detected
✅ 19:55:50 — No threats detected
✅ 19:56:00 — No threats detected
✅ 19:56:10 — No threats detected
✅ 19:56:20 — No threats detected
✅ 19:56:30 — No threats detected
✅ 19:56:40 — No threats detected
✅ 19:56:50 — No threats detected
✅ 19:57:00 — No threats detected
✅ 19:57:10 — No threats detected
✅ 19:57:20 — No threats detected
✅ 19:57:30 — No threats detected
✅ 19:57:40 — No threats detected

🔴 Monitor stopped


In [5]:
import subprocess
import re
from datetime import datetime

def hunt_persistence():
    print(f"\n{'='*55}")
    print(f"🔍 HUNTING: Persistence Mechanisms")
    print('='*55)
    
    # Check systemd services (common persistence location)
    result = subprocess.run(
        ['systemctl', 'list-units', '--type=service', '--state=running'],
        capture_output=True, text=True
    )
    
    # Known legitimate services
    legitimate = [
        'sshd', 'NetworkManager', 'bluetooth', 'cups',
        'docker', 'tor', 'systemd', 'dbus', 'ufw',
        'avahi', 'nftables', 'accounts', 'polkit'
    ]
    
    suspicious = []
    lines = result.stdout.split('\n')
    
    print(f"Running services:")
    for line in lines:
        if '.service' in line:
            service_match = re.search(r'(\S+\.service)', line)
            if service_match:
                service = service_match.group(1)
                is_legit = any(leg in service.lower() for leg in legitimate)
                if is_legit:
                    print(f"  ✅ {service}")
                else:
                    print(f"  ⚠️  {service} ← UNKNOWN — investigate!")
                    suspicious.append(service)
    
    print(f"\nSuspicious services found: {len(suspicious)}")
    return suspicious

suspicious_services = hunt_persistence()


🔍 HUNTING: Persistence Mechanisms
Running services:
  ⚠️  asus-shutdown.service ← UNKNOWN — investigate!
  ⚠️  asusd.service ← UNKNOWN — investigate!
  ✅ bluetooth.service
  ⚠️  bolt.service ← UNKNOWN — investigate!
  ⚠️  containerd.service ← UNKNOWN — investigate!
  ✅ dbus-broker.service
  ✅ docker.service
  ⚠️  NetworkManager.service ← UNKNOWN — investigate!
  ⚠️  nvidia-powerd.service ← UNKNOWN — investigate!
  ⚠️  ollama.service ← UNKNOWN — investigate!
  ✅ polkit.service
  ⚠️  power-profiles-daemon.service ← UNKNOWN — investigate!
  ⚠️  rtkit-daemon.service ← UNKNOWN — investigate!
  ⚠️  sddm.service ← UNKNOWN — investigate!
  ⚠️  supergfxd.service ← UNKNOWN — investigate!
  ✅ systemd-journald.service
  ✅ systemd-logind.service
  ✅ systemd-networkd.service
  ✅ systemd-resolved.service
  ✅ systemd-timesyncd.service
  ✅ systemd-udevd.service
  ✅ systemd-userdbd.service
  ⚠️  udisks2.service ← UNKNOWN — investigate!
  ⚠️  upower.service ← UNKNOWN — investigate!
  ⚠️  user@1000.servi

In [6]:
def hunt_network_connections():
    print(f"\n{'='*55}")
    print(f"🔍 HUNTING: Suspicious Network Connections")
    print('='*55)
    
    # Get all active connections
    result = subprocess.run(
        ['ss', '-tunp'],
        capture_output=True, text=True
    )
    
    # Known legitimate ports
    legitimate_ports = {
        '22': 'SSH', '53': 'DNS', '80': 'HTTP',
        '443': 'HTTPS', '8888': 'Jupyter', '9050': 'Tor',
        '9051': 'Tor Control', '11434': 'Ollama',
        '631': 'CUPS Printing'
    }
    
    print(f"Active network connections:\n")
    lines = result.stdout.split('\n')
    
    suspicious = []
    for line in lines[1:]:  # Skip header
        if not line.strip():
            continue
            
        parts = line.split()
        if len(parts) >= 5:
            local = parts[4]
            port = local.split(':')[-1]
            
            if port in legitimate_ports:
                print(f"  ✅ Port {port:6s} ({legitimate_ports[port]:10s}) → {local}")
            else:
                print(f"  ⚠️  Port {port:6s} (Unknown) → {local}")
                suspicious.append(local)
    
    print(f"\nUnknown connections: {len(suspicious)}")
    return suspicious

hunt_network_connections()


🔍 HUNTING: Suspicious Network Connections
Active network connections:

  ⚠️  Port 68     (Unknown) → 10.61.62.186%wlan0:68
  ⚠️  Port 44933  (Unknown) → 127.0.0.1:44933
  ⚠️  Port 40098  (Unknown) → 127.0.0.1:40098
  ⚠️  Port 40122  (Unknown) → 127.0.0.1:40122
  ✅ Port 8888   (Jupyter   ) → 127.0.0.1:8888
  ✅ Port 8888   (Jupyter   ) → 127.0.0.1:8888
  ⚠️  Port 39540  (Unknown) → 127.0.0.1:39540
  ⚠️  Port 39570  (Unknown) → 127.0.0.1:39570
  ⚠️  Port 60630  (Unknown) → 127.0.0.1:60630
  ⚠️  Port 60684  (Unknown) → 127.0.0.1:60684
  ⚠️  Port 39265  (Unknown) → 127.0.0.1:39265
  ⚠️  Port 60885  (Unknown) → 127.0.0.1:60885
  ✅ Port 8888   (Jupyter   ) → 127.0.0.1:8888
  ⚠️  Port 33403  (Unknown) → 127.0.0.1:33403
  ⚠️  Port 47265  (Unknown) → 127.0.0.1:47265
  ⚠️  Port 56979  (Unknown) → 127.0.0.1:56979
  ⚠️  Port 46505  (Unknown) → 127.0.0.1:46505
  ⚠️  Port 33403  (Unknown) → 127.0.0.1:33403
  ⚠️  Port 44933  (Unknown) → 127.0.0.1:44933
  ⚠️  Port 60885  (Unknown) → 127.0.0.1:60885
  

['10.61.62.186%wlan0:68',
 '127.0.0.1:44933',
 '127.0.0.1:40098',
 '127.0.0.1:40122',
 '127.0.0.1:39540',
 '127.0.0.1:39570',
 '127.0.0.1:60630',
 '127.0.0.1:60684',
 '127.0.0.1:39265',
 '127.0.0.1:60885',
 '127.0.0.1:33403',
 '127.0.0.1:47265',
 '127.0.0.1:56979',
 '127.0.0.1:46505',
 '127.0.0.1:33403',
 '127.0.0.1:44933',
 '127.0.0.1:60885',
 '127.0.0.1:46505',
 '127.0.0.1:55588',
 '127.0.0.1:55592',
 '127.0.0.1:55628',
 '127.0.0.1:47265',
 '127.0.0.1:39080',
 '127.0.0.1:39094',
 '127.0.0.1:50291',
 '127.0.0.1:39265',
 '127.0.0.1:36913',
 '127.0.0.1:42983',
 '127.0.0.1:39265',
 '127.0.0.1:50291',
 '127.0.0.1:46616',
 '127.0.0.1:48463',
 '127.0.0.1:36913',
 '127.0.0.1:52682',
 '127.0.0.1:52696',
 '127.0.0.1:52722',
 '127.0.0.1:35864',
 '127.0.0.1:35872',
 '127.0.0.1:57989',
 '127.0.0.1:57989',
 '127.0.0.1:33896',
 '127.0.0.1:33898',
 '127.0.0.1:33916',
 '127.0.0.1:56979',
 '127.0.0.1:56318',
 '127.0.0.1:56340',
 '127.0.0.1:35734',
 '127.0.0.1:35668',
 '127.0.0.1:38722',
 '127.0.0.1:41

In [7]:
def hunt_external_connections():
    print(f"\n{'='*55}")
    print(f"🔍 HUNTING: External Connections Only")
    print('='*55)
    
    result = subprocess.run(
        ['ss', '-tunp', 'state', 'established'],
        capture_output=True, text=True
    )
    
    print(result.stdout)
    
    # Also check with netstat style
    result2 = subprocess.run(
        ['ss', '-tnp'],
        capture_output=True, text=True
    )
    
    external = []
    for line in result2.stdout.split('\n'):
        if 'ESTAB' in line:
            parts = line.split()
            if len(parts) >= 5:
                peer = parts[4]
                # Filter out localhost
                if not peer.startswith('127.') and \
                   not peer.startswith('[::1]') and \
                   not peer.startswith('[::ffff:127'):
                    print(f"🌐 External: {peer}")
                    external.append(peer)
    
    if not external:
        print("✅ No suspicious external connections found!")
    else:
        print(f"\n⚠️  {len(external)} external connections — investigate!")
    
    return external

hunt_external_connections()


🔍 HUNTING: External Connections Only
Netid Recv-Q Send-Q                             Local Address:Port                          Peer Address:Port Process
udp   0      0                             10.61.62.186%wlan0:68                            10.61.62.101:67   
tcp   0      0                                      127.0.0.1:44933                            127.0.0.1:60684
tcp   0      0                                      127.0.0.1:40098                            127.0.0.1:46505
tcp   0      0                                      127.0.0.1:40122                            127.0.0.1:46505
tcp   0      0                                      127.0.0.1:8888                             127.0.0.1:38722
tcp   0      0                                      127.0.0.1:8888                             127.0.0.1:47074
tcp   0      0                                      127.0.0.1:39540                            127.0.0.1:60885
tcp   0      0                                      127.0.0.1:39570

['[2a03:2880:f314:120:face:b00c:0:167]:443',
 '[64:ff9b::8c52:711a]:443',
 '[2404:6800:4013:813::54]:443',
 '[64:ff9b::226b:f35d]:443',
 '[64:ff9b::226b:f35d]:443',
 '[2607:6bc0::10]:443']

In [8]:
import requests

external_ips = [
    "2a03:2880:f314:120:face:b00c:0:167",
    "64:ff9b::8c52:711a",
    "2404:6800:4013:813::54",
    "64:ff9b::226b:f35d",
    "2607:6bc0::10"
]

print(f"{'='*55}")
print(f"🔍 IDENTIFYING EXTERNAL CONNECTIONS")
print(f"{'='*55}\n")

for ip in external_ips:
    try:
        # Convert IPv6 to IPv4 for lookup where possible
        # 64:ff9b:: prefix means IPv4-mapped IPv6
        if ip.startswith("64:ff9b::"):
            # Extract IPv4 portion
            hex_part = ip.replace("64:ff9b::", "")
            parts = hex_part.split(":")
            if len(parts) == 2:
                oct1 = int(parts[0][:2], 16)
                oct2 = int(parts[0][2:], 16)
                oct3 = int(parts[1][:2], 16)
                oct4 = int(parts[1][2:], 16)
                ipv4 = f"{oct1}.{oct2}.{oct3}.{oct4}"
                lookup_ip = ipv4
            else:
                lookup_ip = ip
        else:
            lookup_ip = ip
            
        response = requests.get(
            f"http://ip-api.com/json/{lookup_ip}",
            timeout=5
        )
        data = response.json()
        
        print(f"IP: {ip}")
        print(f"  Owner:   {data.get('org', 'Unknown')}")
        print(f"  Country: {data.get('country', 'Unknown')}")
        print(f"  City:    {data.get('city', 'Unknown')}")
        
        # Risk assessment
        trusted = ['facebook', 'google', 'mozilla',
                  'cloudflare', 'meta', 'fastly', 'anthropic']
        org = data.get('org', '').lower()
        is_trusted = any(t in org for t in trusted)
        
        if is_trusted:
            print(f"  Status:  ✅ TRUSTED — {data.get('org')}")
        else:
            print(f"  Status:  ⚠️  UNKNOWN — investigate further!")
        print()
        
    except Exception as e:
        print(f"IP: {ip} → Error: {e}\n")

🔍 IDENTIFYING EXTERNAL CONNECTIONS

IP: 2a03:2880:f314:120:face:b00c:0:167
  Owner:   Facebook, Inc
  Country: India
  City:    Mastemau
  Status:  ✅ TRUSTED — Facebook, Inc

IP: 64:ff9b::8c52:711a
  Owner:   GitHub, Inc.
  Country: United States
  City:    San Francisco
  Status:  ⚠️  UNKNOWN — investigate further!

IP: 2404:6800:4013:813::54
  Owner:   Google Public DNS (del)
  Country: India
  City:    Delhi
  Status:  ✅ TRUSTED — Google Public DNS (del)

IP: 64:ff9b::226b:f35d
  Owner:   Google Cloud
  Country: United States
  City:    Kansas City
  Status:  ✅ TRUSTED — Google Cloud

IP: 2607:6bc0::10
  Owner:   Anthropic, PBC
  Country: United States
  City:    San Francisco
  Status:  ✅ TRUSTED — Anthropic, PBC

